# Minimal end-to-end KG generation pipeline

A minimal knowledge-graph generation pipeline built on the
`neo4j_graphrag.pipeline` dataflow DSL, mirroring the stages of a production KG
generation workflow (load → split → embed → extract → write → resolve):

| stage | component | pipeline operator |
| --- | --- | --- |
| Split the document into chunks | `FixedSizeSplitter.split` (sync) | `map_safe` |
| Embed the chunk texts | `TextChunkEmbedder` | `map_async_chunked_safe` |
| Extract entities/relations + lexical graph | `LLMEntityRelationExtractor` | `map_async_chunked_safe` |
| Write the graph to Neo4j | `Neo4jWriter` | `map_async_chunked_safe` |
| Merge duplicate entities | `SinglePropertyExactMatchResolver` | after `collect()` |

The pipeline is a lazy definition over a stream of documents: nothing runs until
`collect()` drains it. Each stage is `*_safe`, so a document that fails is captured
as an `Err` and the remaining documents continue.

**Prerequisites**

- A Neo4j instance at `neo4j://localhost:7687` with user `neo4j` / password `password`
  (from the repo root: `docker compose -f tests/e2e/docker-compose.yml up -d neo4j --wait`).
- `OPENAI_API_KEY` set in the environment for the LLM and the embedder.

See `examples/SETUP.md` for details. To process files instead of inline text, stream
paths and load each with `PdfLoader` as in
`examples/customize/build_graph/pipeline/kg_builder_from_pdf.py`.

In [1]:
import logging

import neo4j

from neo4j_graphrag.components.embedder import TextChunkEmbedder
from neo4j_graphrag.components.entity_relation_extractor import (
    LLMEntityRelationExtractor,
)
from neo4j_graphrag.components.kg_writer import Neo4jWriter
from neo4j_graphrag.components.resolver import SinglePropertyExactMatchResolver
from neo4j_graphrag.components.schema import SchemaBuilder
from neo4j_graphrag.components.text_splitters.fixed_size_splitter import (
    FixedSizeSplitter,
)
from neo4j_graphrag.components.types import LexicalGraphConfig
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.pipeline import Err, Pipeline

logging.basicConfig()
logging.getLogger("neo4j_graphrag").setLevel(logging.INFO)

# Neo4j connection details - update if needed
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")
DATABASE = "neo4j"


/Users/jonnylaw/src/neo4j-graphrag-python/.venv/lib/python3.13/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


/Users/jonnylaw/src/neo4j-graphrag-python/.venv/lib/python3.13/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string


## 1. Text and guiding schema

The schema tells the LLM which node labels, relationship types and
`(start, relationship, end)` triplets to look for in the text. It is built once,
up front - it describes every document in the stream, so it is not a stream stage.

In [2]:
TEXT = """The son of Duke Leto Atreides and the Lady Jessica, Paul is the heir of House Atreides,
an aristocratic family that rules the planet Caladan, the rainy planet, since 10191."""

# Node labels can be plain strings, or dicts with a description and
# properties the LLM will try to fill in.
node_types = [
    "Person",
    {
        "label": "House",
        "description": "Family the person belongs to",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {"label": "Planet", "properties": [{"name": "weather", "type": "STRING"}]},
]
relationship_types = [
    "PARENT_OF",
    {
        "label": "HEIR_OF",
        "description": "Used for inheritor relationship between father and sons",
    },
    {"label": "RULES", "properties": [{"name": "fromYear", "type": "INTEGER"}]},
]
patterns = [
    ("Person", "PARENT_OF", "Person"),
    ("Person", "HEIR_OF", "House"),
    ("House", "RULES", "Planet"),
]

schema = await SchemaBuilder().run(
    node_types=node_types,
    relationship_types=relationship_types,
    patterns=patterns,
)


INFO:neo4j_graphrag.components.schema:Converting string 'Person' to NodeType with default 'name' property and additional_properties=True to allow flexible property extraction.


## 2. Pipeline definition

Each component's `run` method becomes one stream stage. The extractor runs with
`create_lexical_graph=True` (the default), so its output graph contains both the
lexical graph (`Document`/`Chunk` nodes, `NEXT_CHUNK` relationships) and the
extracted entities linked back to their chunks - a single `Neo4jWriter` call
persists both.

Splitting is pure string slicing, so `FixedSizeSplitter` exposes it as the
synchronous `split` method and the stage is a plain `map_safe`. The I/O-bound stages
(embed, extract, write) are `map_async_chunked_safe`. In both cases per-document
failures are captured as `Err` values and collected by the `on_error` handler instead
of aborting the run.

In [3]:
lexical_graph_config = LexicalGraphConfig()
# 'path' identifies the Document node the chunks are attached to
document_info = {"path": "dune.txt"}

# gpt-5 counts reasoning tokens against max_completion_tokens, so too small a
# budget returns empty content; reasoning_effort="low" keeps the cost of this
# small example down.
LLM_MODEL_PARAMS = {
    "max_completion_tokens": 16000,
    "reasoning_effort": "low",
    "response_format": {"type": "json_object"},
}


def build_pipeline(driver: neo4j.Driver) -> Pipeline:
    splitter = FixedSizeSplitter(chunk_size=500, chunk_overlap=50)
    # OpenAIEmbeddings wraps the *sync* OpenAI client, which does not bind to
    # an event loop, so a single instance can be shared across stages.
    chunk_embedder = TextChunkEmbedder(embedder=OpenAIEmbeddings())
    writer = Neo4jWriter(driver)

    async def extract(chunks):
        # OpenAILLM wraps an async httpx client, which binds to the event loop
        # of the stage that uses it - so it is created and closed inside the
        # stage function rather than shared from the notebook's own loop.
        async with OpenAILLM(
            model_name="gpt-5", model_params=LLM_MODEL_PARAMS
        ) as llm:
            extractor = LLMEntityRelationExtractor(llm=llm)
            return await extractor.run(
                chunks=chunks,
                document_info=document_info,
                lexical_graph_config=lexical_graph_config,
                schema=schema,
            )

    errors: list[Err] = []
    pipeline = (
        Pipeline([TEXT], label="documents")
        .map_safe(splitter.split, label="split")
        .map_async_chunked_safe(chunk_embedder.run, label="embed")
        .map_async_chunked_safe(extract, label="extract")
        .map_async_chunked_safe(
            lambda graph: writer.run(
                graph=graph, lexical_graph_config=lexical_graph_config
            ),
            label="write",
        )
        .on_error(errors.append)
    )
    return pipeline, errors


## 3. Run the pipeline

`collect()` drains the stream: every document is split, embedded, extracted and
written. `LocalInterpreter` evaluates the async stages blocking - in a notebook,
where an event loop is already running, chunks are driven on a dedicated
event-loop thread, so `collect()` works unchanged from a cell.

Entity resolution runs afterwards: it merges duplicate entities already written
to Neo4j, so it is a barrier over the whole stream rather than a per-document
stage.

In [ ]:
with neo4j.GraphDatabase.driver(URI, auth=AUTH) as driver:
    pipeline, errors = build_pipeline(driver)
    results = pipeline.collect()
    resolution = await SinglePropertyExactMatchResolver(driver).run()

print(f"documents written: {len(results)}, failed: {len(errors)}")
resolution


documents written: 1, failed: 0


ResolutionStats(number_of_nodes_to_resolve=5, number_of_created_nodes=4)

## 4. Inspect the generated graph

The entity graph (nodes with the schema labels) and the lexical graph
(`Document`/`Chunk` nodes, `NEXT_CHUNK` relationships) now live side by side in
Neo4j, with duplicate entities merged by the resolver.

In [ ]:
with neo4j.GraphDatabase.driver(URI, auth=AUTH) as driver:
    entities, _, _ = driver.execute_query(
        "MATCH (n) WHERE NOT n:Document AND NOT n:Chunk "
        "RETURN labels(n) AS labels, n.name AS name "
        "ORDER BY labels, name",
        database_=DATABASE,
    )
    for record in entities:
        print(record["labels"], record["name"])

    rels, _, _ = driver.execute_query(
        "MATCH (a)-[r]->(b) "
        "WHERE NOT a:Document AND NOT a:Chunk AND NOT b:Document AND NOT b:Chunk "
        "RETURN a.name AS source, type(r) AS type, b.name AS target",
        database_=DATABASE,
    )
    for record in rels:
        print(record["source"], "-", record["type"], "->", record["target"])


['__KGBuilder__', 'Person', '__Entity__'] Jessica
['__KGBuilder__', 'Person', '__Entity__'] Leto Atreides
['__KGBuilder__', 'Person', '__Entity__'] Paul
['__KGBuilder__', '__Entity__', 'House'] House Atreides
['__KGBuilder__', '__Entity__', 'Planet'] None
Leto Atreides - PARENT_OF -> Paul
Jessica - PARENT_OF -> Paul
Paul - HEIR_OF -> House Atreides
House Atreides - RULES -> None
